In [16]:
library(jsonlite)
library(dplyr)
library(purrr)

fetch_openmeteo_wind_point <- function(lat, lon) {
  url <- sprintf(
    paste0(
      "https://api.open-meteo.com/v1/forecast?",
      "latitude=%f&longitude=%f",
      "&current=wind_speed_10m,wind_direction_10m",
      "&wind_speed_unit=ms"
    ),
    lat, lon
  )

  x <- jsonlite::fromJSON(url)

  if (is.null(x$current) ||
      is.null(x$current$wind_speed_10m) ||
      is.null(x$current$wind_direction_10m)) {
    return(data.frame(
      lat = numeric(0),
      lng = numeric(0),
      speed_mps = numeric(0),
      direction_deg = numeric(0)
    ))
  }

  data.frame(
    lat = lat,
    lng = lon,
    speed_mps = x$current$wind_speed_10m,
    direction_deg = x$current$wind_direction_10m
  )
}

build_static_wind_field <- function(
  lats = seq(35.9, 37.7, by = 0.2),
  lngs = seq(-123.4, -121.1, by = 0.2),
  arrow_km = 15
) {
  pts <- expand.grid(lat = lats, lng = lngs)

  wind <- purrr::map2_dfr(pts$lat, pts$lng, fetch_openmeteo_wind_point)

  if (nrow(wind) == 0) {
    stop("No wind data returned from Open-Meteo")
  }

  # meteorological direction is where wind comes from
  # arrows should point toward where the wind is going
  dir_toward <- (wind$direction_deg + 180) %% 360
  angle_rad <- (90 - dir_toward) * pi / 180

  max_speed <- max(wind$speed_mps, na.rm = TRUE)
  if (!is.finite(max_speed) || max_speed <= 0) max_speed <- 1

  wind <- wind |>
    mutate(
      dlat = (speed_mps / max_speed) * (arrow_km / 111) * sin(angle_rad),
      dlng = (speed_mps / max_speed) * (arrow_km / (111 * cos(lat * pi / 180))) * cos(angle_rad),
      lat2 = lat + dlat,
      lng2 = lng + dlng
    )

  wind
}

In [17]:
wind <- build_static_wind_field(
  lats = seq(36.0, 37.6, by = 0.3),
  lngs = seq(-123.3, -121.2, by = 0.3),
  arrow_km = 12
)

head(wind)

,lat,lng,speed_mps,direction_deg,dlat,dlng,lat2,lng2
,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
1,36.0,-123.3,5.83,329,-0.06864639,0.05098399,35.93135,-123.2490
2,36.3,-123.3,6.02,324,-0.06690178,0.06031180,36.23310,-123.2397
3,36.6,-123.3,6.22,323,-0.06823741,0.06405015,36.53176,-123.2359
4,36.9,-123.3,6.03,320,-0.06345339,0.06658089,36.83655,-123.2334
5,37.2,-123.3,5.87,314,-0.05601353,0.07282050,37.14399,-123.2272
6,37.5,-123.3,5.87,316,-0.05800370,0.07060350,37.44200,-123.2294


In [18]:
nrow(wind)

[1] 48

In [19]:
ncol(wind)

[1] 8

In [20]:
colnames(wind)

[1] "lat"           "lng"           "speed_mps"     "direction_deg"
[5] "dlat"          "dlng"          "lat2"          "lng2"

In [22]:
saveRDS(wind, "/home/benjamin/Projects/santa_cruz_waves/data/raw/wind_latest.rds")